# 03. 智能语音机器人：语音→思考→语音闭环

把前面两章的能力串联起来，做一个能「听懂你说话 → 生成回复 → 说给你听」的机器人。

**流水线架构**（三段式）：

```
用户语音 ──▶ Whisper large-v3 ──▶ 中文文本
                                       │
                                       ▼
                                  商汤 glm-5.2 ──▶ 回复文本
                                                  │
                                                  ▼
                       XTTS-v2（吴彦祖音色）◀──────┘
                         合成语音 ◀──────  文字
```

**本章重点**：用 Gradio 6 搭一个网页界面，麦克风录音 → 自动转文字 → LLM 思考 → 生成语音播放，全程零手工切文件。

In [1]:
# Cell 1: 环境准备 —— 兼容性补丁 + 全量导入
import os, json, time, io, numpy as np, soundfile as sf

# ── coqui-tts 0.27.5 + transformers 5.x 兼容补丁 ────────────────────────────
import transformers.pytorch_utils as _tpu, torch
if not hasattr(_tpu, "isin_mps_friendly"):
    _tpu.isin_mps_friendly = torch.isin

# ── 语音识别 Whisper ───────────────────────────────────────────────────────
from transformers import pipeline
import torch

# ── 语音合成 XTTS-v2 ────────────────────────────────────────────────────────
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# ── 商汤 LLM（OpenAI 兼容协议） ──────────────────────────────────────────────
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# ── Gradio 界面 ────────────────────────────────────────────────────────────
import gradio as gr

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("gradio:", gr.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

S:\PythonAILearningCourse\15-transformers-basics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.5.1+cu121 | CUDA: True
gradio: 6.27.0
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 1. 加载 Whisper large-v3 本地模型

Whisper 是 OpenAI 开源的多语言语音识别模型，v3-large 是目前最强版本：

- **20+ 种语言**原生支持，含中文普通话 / 粤语
- **3GB** 权重，RTX 4060 上 fp16 可跑
- 本地路径 `P:\learn tools\models\whisper-large-v3` 已预置完整权重（`model.safetensors` + `pytorch_model.bin` 都在）

用 transformers Pipeline 最省事：自动组装 processor + model，`device="cuda"` 走 GPU 加速。

In [2]:
# Cell 2: 加载 Whisper large-v3（首次约 1~2 分钟读权重）
WHISPER_DIR = r"P:\learn tools\models\whisper-large-v3"

print("正在加载 Whisper large-v3 ...")
t0 = time.time()
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_DIR,
    device="cuda" if torch.cuda.is_available() else "cpu",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
print("Whisper 加载完成，耗时 %.1f 秒" % (time.time() - t0))

# 预热：用 0.5s 空白音频跑一次，让 CUDA kernel 初始化
dummy = np.zeros((80000,), dtype=np.float32)  # 5 秒静音
_ = whisper_pipe(dummy, generate_kwargs={"language": "zh", "max_new_tokens": 10})
print("Whisper 预热完成")

正在加载 Whisper large-v3 ...


Loading weights: 100%|██████████| 1259/1259 [00:00<00:00, 3762.31it/s]


Whisper 加载完成，耗时 10.0 秒


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.ge

Whisper 预热完成


## 2. 加载 XTTS-v2 + 预提取吴彦祖音色

沿用 02 notebook 的本地加载三步法，一次性把参考音频的条件向量提取好，后续每轮对话复用，省掉每次推理都要算 Speaker Encoder 的开销。

输出采样率务必用 `config.model_args["output_sample_rate"]`（24000Hz），不是历史遗留的 `config.audio.sample_rate`（22050Hz）。

In [3]:
# Cell 3: 加载 XTTS-v2 + 预提取参考音色
MODEL_DIR = r"S:\PythonAILearningCourse\08-tts-voice-robot\model\XTTS-v2"
REF_WAV  = r"S:\PythonAILearningCourse\08-tts-voice-robot\02.coqui-tts\wyz_ref_22k.wav"

print("正在加载 XTTS-v2 ...")
t0 = time.time()
config = XttsConfig()
config.load_json(os.path.join(MODEL_DIR, "config.json"))
tts_model = Xtts.init_from_config(config)
tts_model.load_checkpoint(config, checkpoint_dir=MODEL_DIR, eval=True)
tts_model.cuda()
print("XTTS-v2 加载完成，耗时 %.1f 秒" % (time.time() - t0))

# 预提取条件向量：一次算，全程复用
print("正在提取参考音频音色向量 ...")
gpt_cond_latent, speaker_embedding = tts_model.get_conditioning_latents(audio_path=REF_WAV)
print("GPT latent:", tuple(gpt_cond_latent.shape), "| 说话人向量:", tuple(speaker_embedding.shape))
SR_OUT = config.model_args["output_sample_rate"]  # 24000
print("输出采样率:", SR_OUT, "Hz")

正在加载 XTTS-v2 ...


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.


XTTS-v2 加载完成，耗时 18.4 秒
正在提取参考音频音色向量 ...
GPT latent: (1, 32, 1024) | 说话人向量: (1, 512, 1)
输出采样率: 24000 Hz


## 3. 配置商汤 LLM

商汤 SenseNova 平台提供 OpenAI 兼容的 Chat API，base_url 固定为 `https://token.sensenova.cn/v1`。

推荐用 **glm-5.2**（不是 sensenova-6.8-flash-lite）：后者是长推理模型，会把 token 预算耗尽在思考过程上，正文 content 为空。glm-5.2 是推理模型里收敛最快的，做短对话回复最合适。

API Key 已写入代码（来自之前项目），生产环境应放环境变量。

In [4]:
# Cell 4: 配置商汤 LLM —— 带 TPM 限流自动重试
import random

LLM_API_KEY = os.getenv('SENSENOVA_API_KEY', '')
LLM_BASE_URL = "https://token.sensenova.cn/v1"
LLM_MODEL    = "glm-5.2"

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    temperature=0.7,
    request_timeout=30,
)

# 简单重试装饰器：商汤 API 有全局 TPM 限流，约 5 次连续调用后触发 429
def call_llm(messages, max_retries=3, base_delay=60):
    for attempt in range(max_retries):
        try:
            resp = llm.invoke(messages)
            content = resp.content if resp.content else (
                resp.additional_kwargs.get("reasoning_content", "") or "（模型返回为空）"
            )
            return content.strip()
        except Exception as e:
            err = str(e)
            if "429" in err or "RateLimit" in err or "TPM" in err:
                delay = base_delay * (2 ** attempt) + random.uniform(0, 10)
                print("[LLM] 触发限流，%.0f 秒后重试 (%d/%d)" % (delay, attempt + 1, max_retries))
                time.sleep(delay)
            else:
                raise
    raise RuntimeError("LLM 重试 %d 次仍失败" % max_retries)

# 预热一次，验证 API 通畅
resp = call_llm([
    SystemMessage(content="你是参考音频里那个人风格的语音助手，回答简洁，30字以内。"),
    HumanMessage(content="你好，介绍一下自己。"),
])
print("LLM 预热回复:", resp)

LLM 预热回复: 你好，我是你的专属语音助手。随时待命，有事请直说。


## 4. 核心流水线函数

输入一段音频（Gradiu Audio 组件返回的 numpy 数组或 wav 文件路径），依次走完 Whisper → 商汤 → XTTS，返回：

- **文字回复**（用于 UI 展示）
- **合成音频**（用于 Gradio Audio 组件播放）

同时维护一段对话历史（最多保留最近 6 轮），让 LLM 能记住上下文，回答连贯。

In [5]:
# Cell 5: 核心流水线 —— 语音输入 → 文字回复 + 语音输出

# 对话历史：System + 最近 6 轮
SYSTEM_PROMPT = (
    "你是一个名叫「小祖」的中文语音助手，说话风格像参考音频里的那个人——"
    "自然、自信、带点从容的幽默。回答要简洁明了，控制在80字以内。"
)

conversation_history = []  # [(role, text), ...]

def reset_chat():
    '''清除对话历史'''
    conversation_history.clear()
    return [], ""

def run_pipeline(audio_input):
    '''
    输入: audio_input 可能是
      - str: wav 文件路径
      - (sr, ndarray): Gradio Audio 组件的 numpy 元组
    输出:
      - reply_text: 商汤的文字回复 (str)
      - reply_audio: XTTS 合成的 (sr, wav) 给 Gradio Audio 播放
      - transcript: Whisper 识别出的用户说的话 (str)
    '''
    if audio_input is None:
        return "", None, ""

    # ── Step 1: Whisper 语音识别 ──────────────────────────────────────────
    print("[1/3] Whisper 语音识别 ...")
    t0 = time.time()

    if isinstance(audio_input, str):
        # 文件路径
        audio, sr = sf.read(audio_input)
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != 16000:
            import librosa
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=16000)
            sr = 16000
    elif isinstance(audio_input, tuple):
        # (sr, ndarray) from Gradio Audio source
        sr, audio = audio_input
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != 16000:
            import librosa
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=16000)
            sr = 16000
    else:
        return "不支持的输入格式", None, ""

    result = whisper_pipe(
        audio.astype(np.float32),
        generate_kwargs={"language": "zh", "task": "transcribe", "max_new_tokens": 256},
    )
    transcript = result["text"].strip()
    print("  Whisper 识别结果:「%s」| 耗时 %.1f 秒" % (transcript, time.time() - t0))

    if not transcript:
        return "没听清你说什么，请再说一遍。", None, ""

    # ── Step 2: 商汤 LLM 对话 ────────────────────────────────────────────
    print("[2/3] 商汤 LLM 思考 ...")
    t1 = time.time()

    conversation_history.append(("human", transcript))
    # 构造 messages，带历史但只保留最近 6 轮
    messages = [SystemMessage(content=SYSTEM_PROMPT)]
    for role, text in conversation_history[-6:]:
        if role == "human":
            messages.append(HumanMessage(content=text))
        else:
            messages.append(SystemMessage(content=text))  # 用 System 表示 assistant 回复也行

    reply_text = call_llm(messages)
    conversation_history.append(("ai", reply_text))
    print("  LLM 回复:「%s」| 耗时 %.1f 秒" % (reply_text, time.time() - t1))

    # ── Step 3: XTTS-v2 语音合成 ──────────────────────────────────────────
    print("[3/3] XTTS-v2 合成语音 ...")
    t2 = time.time()

    out = tts_model.inference(
        text=reply_text,
        language="zh-cn",
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
        temperature=0.65,
        top_p=0.5,
        top_k=50,
        repetition_penalty=5.0,
        speed=1.0,
        enable_text_splitting=True,
    )
    wav = out["wav"]
    print("  XTTS 合成 %.1f 秒音频 | 耗时 %.1f 秒" % (len(wav) / SR_OUT, time.time() - t2))
    print("  全链路总耗时 %.1f 秒" % (time.time() - t0))

    return reply_text, (SR_OUT, wav), transcript

## 5. Gradio 界面

Gradio 6.x 的 Audio 组件：

- `source="microphone"`：从浏览器麦克风实时录音，点击说话、松手提交
- `source="upload"`：也可上传 wav 文件测试
- `type="numpy"`：返回 (sample_rate, ndarray) 元组，方便直接喂 pipeline
- 输出 Audio 直接给 `(sr, wav)` 元组即可播放

再加一个 Textbox 显示 Whisper 识别结果和 LLM 回复，方便调试。

In [6]:
# Cell 6: Gradio 界面搭建

demo = gr.Blocks(title="智能语音机器人 · 小祖")

with demo:
    gr.Markdown('''
    # 🎙️ 智能语音机器人「小祖」
    对着麦克风说话 → Whisper 识别 → 商汤思考 → XTTS 用 wyz 音色回复
    ''')

    with gr.Row():
        with gr.Column():
            audio_in = gr.Audio(
                label="🎤 说话",
                sources="microphone",
                type="numpy",
            )
            upload_in = gr.Audio(
                label="📎 或上传音频文件",
                sources="upload",
                type="numpy",
            )
        with gr.Column():
            transcript_box = gr.Textbox(label="👂 小祖听到了", lines=2, interactive=False)
            reply_box = gr.Textbox(label="🧠 小祖想了想", lines=3, interactive=False)
            audio_out = gr.Audio(label="🔊 小祖说", autoplay=True)

    with gr.Row():
        btn_mic = gr.Button("🎙️ 提交麦克风录音", variant="primary")
        btn_upload = gr.Button("📎 提交上传音频")
        btn_reset = gr.Button("🗑️ 重置对话")

    btn_mic.click(
        run_pipeline,
        inputs=audio_in,
        outputs=[reply_box, audio_out, transcript_box],
    )
    btn_upload.click(
        run_pipeline,
        inputs=upload_in,
        outputs=[reply_box, audio_out, transcript_box],
    )
    btn_reset.click(
        reset_chat,
        outputs=[transcript_box, reply_box],
    )

print("Gradio 界面组件搭建完成")

Gradio 界面组件搭建完成


In [7]:
# Cell 7: 启动 Gradio 服务
# share=False 则仅本机可访问（默认 http://127.0.0.1:7860）
# 有需要内网穿透可改 share=True，Gradio 会给一个 gradio.live 临时公网地址

demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=False,
)

* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


## 6. 小结

**三段式流水线**：Whisper（本地 ASR） → 商汤 LLM（云端思考） → XTTS-v2（本地 TTS）

- **Whisper large-v3**：3GB 本地模型，中文识别 98%+ 准确率，pipeline 开箱即用
- **商汤 glm-5.2**：OpenAI 兼容协议，推理模型收敛比 sensenova-6.8-flash-lite 更稳
- **XTTS-v2**：预提取条件向量省 GPU 开销，24kHz 真实输出采样率
- **Gradio 6**：Audio 组件麦克风录音 + 自动播放回复，全程语音闭环

**优化方向**：
1. Whisper 换 faster-whisper（CTranslate2 量化），速度翻倍、显存减半
2. 加流式输出：商汤用 `.stream()` 逐 token 返回，XTTS 也有流式 API
3. 多音色切换：预置多段参考音频，下拉菜单选不同"人格"
4. 语音打断检测（VAD）+ 对话状态机：让机器人主动接话而不是只被动等待